# 03 — No-show risk model (Part 3)



In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

In [12]:
df = pd.read_csv('/content/careloop-03 clean /appointments_clean.csv')
appointment_ids = df['appointment_id'].copy()
y = df['no_show'].astype(int)
X = df[['age','distance_km','fee_inr','lead_days','prior_no_shows','reminder_sent','booking_channel','visit_type']].copy()
X['reminder_sent'] = X['reminder_sent'].map({'Yes':1,'No':0})
X = pd.get_dummies(X, columns=['booking_channel','visit_type'], dtype=int)
print('Feature count:', X.shape[1], 'columns:', list(X.columns))
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
print('Train:',len(X_train),'Test:',len(X_test),'Test no-shows:',int(y_test.sum()))
print('Always Attended baseline accuracy:', accuracy_score(y_test, np.zeros(len(y_test),dtype=int)))

Feature count: 11 columns: ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
Train: 792 Test: 198 Test no-shows: 49
Always Attended baseline accuracy: 0.7525252525252525


## 3.2 Logistic regression



In [11]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
lr = LogisticRegression(max_iter=1000).fit(X_train_s,y_train)
lr_pred = lr.predict(X_test_s)
lr_prob = lr.predict_proba(X_test_s)[:,1]
print('Accuracy:',accuracy_score(y_test,lr_pred))
print('Confusion matrix:\n',confusion_matrix(y_test,lr_pred))
print('Precision:',precision_score(y_test,lr_pred,zero_division=0))
print('Recall:',recall_score(y_test,lr_pred,zero_division=0))
print('F1:',f1_score(y_test,lr_pred,zero_division=0))
print('ROC-AUC:',roc_auc_score(y_test,lr_prob))
for name,coef in zip(X.columns,lr.coef_[0]): print(name,coef)
rem_idx = list(X.columns).index('reminder_sent')
rem_coef = lr.coef_[0][rem_idx]
print('exp(scaled coefficient):',np.exp(rem_coef))
print('exp(coefficient / SD), actual No-to-Yes change:',np.exp(rem_coef/scaler.scale_[rem_idx]))

Accuracy: 0.7878787878787878
Confusion matrix:
 [[139  10]
 [ 32  17]]
Precision: 0.6296296296296297
Recall: 0.3469387755102041
F1: 0.4473684210526316
ROC-AUC: 0.788933022873579
age -0.2538801442716842
distance_km 0.25344770956317536
fee_inr 0.4527614630856424
lead_days 0.6144707304969083
prior_no_shows 0.6496116219621334
reminder_sent -0.5806686033659922
booking_channel_App -0.04680357040738989
booking_channel_Phone 0.2158813749282703
booking_channel_Walk-In -0.18636416836300682
visit_type_Consultation 0.01565801179835534
visit_type_Follow-up -0.015658011798355072
exp(scaled coefficient): 0.559524141750829
exp(coefficient / SD), actual No-to-Yes change: 0.3093980903124433


## 3.3 Decision trees: compare overfitting

In [5]:
tree_full = DecisionTreeClassifier(random_state=42).fit(X_train,y_train)
tree3 = DecisionTreeClassifier(max_depth=3,random_state=42).fit(X_train,y_train)
for label,model in [('Full tree',tree_full),('Depth-3 tree',tree3)]:
    print(label,'train accuracy=',accuracy_score(y_train,model.predict(X_train)),
          'test accuracy=',accuracy_score(y_test,model.predict(X_test)))

Full tree train accuracy= 1.0 test accuracy= 0.6717171717171717
Depth-3 tree train accuracy= 0.821969696969697 test accuracy= 0.7777777777777778


## 3.4 Random forest and comparison

In [6]:
rf = RandomForestClassifier(n_estimators=100,random_state=42).fit(X_train,y_train)
rf_pred = rf.predict(X_test); rf_prob = rf.predict_proba(X_test)[:,1]
print('RF accuracy:',accuracy_score(y_test,rf_pred),'recall:',recall_score(y_test,rf_pred), 'AUC:',roc_auc_score(y_test,rf_prob))
print('Top 3 importances:', sorted(zip(X.columns,rf.feature_importances_),key=lambda z:z[1],reverse=True)[:3])
for label,model,prob in [('Logistic',lr,lr_prob),('Depth-3 tree',tree3,tree3.predict_proba(X_test)[:,1]),('Random forest',rf,rf_prob)]:
    pred = (prob >= .5).astype(int)
    print(label,'accuracy',accuracy_score(y_test,pred),'recall',recall_score(y_test,pred),'AUC',roc_auc_score(y_test,prob))

RF accuracy: 0.7474747474747475 recall: 0.30612244897959184 AUC: 0.7057252431173813
Top 3 importances: [('lead_days', np.float64(0.21990547110065853)), ('age', np.float64(0.19678780126634898)), ('distance_km', np.float64(0.19473924793399736))]
Logistic accuracy 0.7878787878787878 recall 0.3469387755102041 AUC 0.788933022873579
Depth-3 tree accuracy 0.7777777777777778 recall 0.22448979591836735 AUC 0.6996986714148747
Random forest accuracy 0.7575757575757576 recall 0.3469387755102041 AUC 0.7057252431173813


## 3.5 Choose using training cross-validation.



In [7]:
lr_pipe = Pipeline([('scaler',StandardScaler()),('model',LogisticRegression(max_iter=1000))])
tree3_model = DecisionTreeClassifier(max_depth=3,random_state=42)
rf_base = RandomForestClassifier(random_state=42)
grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    {'n_estimators':[100,200],'max_depth':[3,5,8]},cv=5,scoring='roc_auc')
grid.fit(X_train,y_train)
print('Tuned RF params:',grid.best_params_,'CV AUC:',grid.best_score_)
models = {'Logistic pipeline':lr_pipe,'Depth-3 tree':tree3_model,'Random forest':rf_base,'Tuned forest':grid.best_estimator_}
cv_scores = {}
for name,model in models.items():
    cv_scores[name] = cross_val_score(model,X_train,y_train,cv=5,scoring='roc_auc').mean()
print('Training CV AUC:',cv_scores)
# choose highest from training CV only
winner_name = max(cv_scores,key=cv_scores.get)
winner = models[winner_name]
print('Winner:',winner_name)

Tuned RF params: {'max_depth': 3, 'n_estimators': 200} CV AUC: 0.7673242251739405
Training CV AUC: {'Logistic pipeline': np.float64(0.7737139899948249), 'Depth-3 tree': np.float64(0.6784494968661952), 'Random forest': np.float64(0.7466024782933701), 'Tuned forest': np.float64(0.7673242251739405)}
Winner: Logistic pipeline


In [8]:
oof_prob = cross_val_predict(winner,X_train,y_train,cv=5,method='predict_proba')[:,1]
for threshold in [0.5,0.35]:
    pred = (oof_prob >= threshold).astype(int)
    print('threshold',threshold,'precision',precision_score(y_train,pred,zero_division=0),
          'recall',recall_score(y_train,pred,zero_division=0),'F1',f1_score(y_train,pred,zero_division=0),
          'flagged',int(pred.sum()))
# Brief's expected choice is threshold 0.35 because its OOF F1 is higher.
threshold = 0.35
winner.fit(X_train,y_train)
test_prob = winner.predict_proba(X_test)[:,1]
test_flag = (test_prob >= threshold).astype(int)
print('Final test precision:',precision_score(y_test,test_flag,zero_division=0))
print('Final test recall:',recall_score(y_test,test_flag,zero_division=0))
print('Final test flagged:',int(test_flag.sum()))
print('Final test ROC-AUC:',roc_auc_score(y_test,test_prob))
risk = pd.DataFrame({'appointment_id':appointment_ids.loc[X_test.index].values,
                     'no_show':y_test.values,'risk_prob':test_prob,'flag':test_flag})
risk.to_csv('test_risk_scores.csv',index=False)
print('Saved test_risk_scores.csv with',len(risk),'rows')

threshold 0.5 precision 0.5753424657534246 recall 0.2485207100591716 F1 0.34710743801652894 flagged 73
threshold 0.35 precision 0.512987012987013 recall 0.46745562130177515 F1 0.4891640866873065 flagged 154
Final test precision: 0.5714285714285714
Final test recall: 0.5714285714285714
Final test flagged: 49
Final test ROC-AUC: 0.788933022873579
Saved test_risk_scores.csv with 198 rows
